# 🎯 K-Means Clustering — End-to-End ML Notebook

---

## Overview

**Topic:** K-Means Clustering — Unsupervised Learning  
**Dataset:** [Mall Customer Segmentation Data — Kaggle](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)  
**Difficulty:** Beginner → Intermediate

### What You Will Learn
- How K-Means works from the ground up (NumPy only)
- How to use Scikit-Learn's optimized implementation
- How to preprocess data, evaluate clusters, and tune hyperparameters
- How to interpret the Elbow Method and Silhouette Score
- The top interview questions on K-Means — with model answers

### Dataset Description
The **Mall Customers** dataset contains 200 customers with:
- `CustomerID` — unique identifier
- `Gender` — Male / Female
- `Age` — customer age
- `Annual Income (k$)` — annual income in thousands of USD
- `Spending Score (1–100)` — mall-assigned score based on purchasing behavior

> **Goal:** Segment customers into meaningful groups so marketing campaigns can be targeted more effectively.


---
## Cell 1b — Load & Explore the Dataset

We load the Mall Customers CSV directly. If running on Kaggle, use:
```
/kaggle/input/customer-segmentation-tutorial-in-python/Mall_Customers.csv
```
Otherwise download from the link above and place in your working directory.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Load dataset ──────────────────────────────────────────────────────────────
# Download from: https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
# Place Mall_Customers.csv in the same folder as this notebook

try:
    df = pd.read_csv('Mall_Customers.csv')
except FileNotFoundError:
    # Fallback: generate synthetic data with the same schema for demo purposes
    np.random.seed(42)
    n = 200
    df = pd.DataFrame({
        'CustomerID': range(1, n+1),
        'Genre': np.random.choice(['Male', 'Female'], n),
        'Age': np.random.randint(18, 70, n),
        'Annual Income (k$)': np.random.randint(15, 140, n),
        'Spending Score (1-100)': np.random.randint(1, 100, n)
    })
    print("⚠️  Mall_Customers.csv not found — using synthetic data with same schema.")
    print("    Download from: https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python\n")

print(f"Shape: {df.shape}")
print("\n── First 5 rows ──")
display(df.head())


In [ ]:
print("── DataFrame Info ──")
df.info()
print("\n── Statistical Summary ──")
display(df.describe().round(2))


In [ ]:
# Check for missing values
print("── Missing Values ──")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")
print("\n── Value Counts: Genre ──")
print(df['Genre'].value_counts())


---
## Cell 2 — Theory Recap

### K-Means: The Core Idea
K-Means partitions `n` data points into `K` non-overlapping clusters by minimizing the **Within-Cluster Sum of Squares (WCSS)**, also called **inertia**.

### Objective Function

$$J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \| x_i - \mu_k \|^2$$

| Symbol | Meaning |
|--------|---------|
| `J` | Total inertia — what we minimize |
| `K` | Number of clusters (your hyperparameter) |
| `C_k` | Set of points assigned to cluster `k` |
| `x_i` | A single data point (feature vector) |
| `μ_k` | Centroid (mean) of cluster `k` |
| `‖·‖²` | Squared Euclidean distance |

### Euclidean Distance
$$d(p, q) = \sqrt{\sum_{j=1}^{d}(p_j - q_j)^2}$$

### The Algorithm Loop
1. **Initialize** K centroids (randomly or via K-Means++)
2. **Assign** each point to the nearest centroid
3. **Update** each centroid to the mean of its assigned points
4. **Repeat** steps 2–3 until centroids stop moving (convergence)

### Convergence
The algorithm converges when centroid positions change by less than a tolerance `tol` between iterations — or after `max_iter` iterations, whichever comes first.

> **Key insight:** K-Means is guaranteed to converge but only to a *local* minimum. Running with multiple random initializations (`n_init > 1`) mitigates this.


---
## Cell 3 — From-Scratch Implementation (NumPy Only)

Before reaching for Scikit-Learn, we build K-Means from the ground up using only NumPy.  
This makes every step of the algorithm transparent and debuggable.

**Steps implemented:**
1. Random centroid initialization
2. Euclidean distance computation (vectorized)
3. Cluster assignment via `argmin`
4. Centroid update via `np.mean`
5. Convergence check via centroid shift threshold


In [ ]:
import numpy as np

class KMeansScratch:
    """
    K-Means Clustering implemented from scratch using NumPy only.
    Matches the Scikit-Learn API: fit(), predict(), fit_predict().
    """

    def __init__(self, n_clusters=3, max_iter=300, tol=1e-4, random_state=42):
        self.n_clusters   = n_clusters
        self.max_iter     = max_iter
        self.tol          = tol
        self.random_state = random_state
        self.centroids_   = None
        self.labels_      = None
        self.inertia_     = None
        self.n_iter_      = 0

    def _euclidean(self, X, centroids):
        """Compute distance from every point to every centroid. Returns (n, K) matrix."""
        # X: (n, d), centroids: (K, d)  →  distances: (n, K)
        diff = X[:, np.newaxis, :] - centroids[np.newaxis, :, :]   # (n, K, d)
        return np.sqrt((diff ** 2).sum(axis=2))                     # (n, K)

    def fit(self, X):
        rng = np.random.RandomState(self.random_state)

        # Step 1 — Random initialization: pick K points as starting centroids
        idx = rng.choice(len(X), size=self.n_clusters, replace=False)
        self.centroids_ = X[idx].copy()

        for iteration in range(self.max_iter):
            # Step 2 — Assignment: assign each point to nearest centroid
            distances = self._euclidean(X, self.centroids_)  # (n, K)
            labels    = np.argmin(distances, axis=1)          # (n,)

            # Step 3 — Update: move each centroid to mean of its cluster
            new_centroids = np.array([
                X[labels == k].mean(axis=0) if (labels == k).sum() > 0
                else self.centroids_[k]       # handle empty cluster
                for k in range(self.n_clusters)
            ])

            # Step 4 — Convergence check
            shift = np.linalg.norm(new_centroids - self.centroids_)
            self.centroids_ = new_centroids
            self.n_iter_    = iteration + 1

            if shift < self.tol:
                print(f"  ✅ Converged after {self.n_iter_} iterations (centroid shift={shift:.6f})")
                break

        self.labels_ = labels
        # Compute inertia (WCSS)
        self.inertia_ = sum(
            ((X[labels == k] - self.centroids_[k]) ** 2).sum()
            for k in range(self.n_clusters)
        )
        return self

    def predict(self, X):
        distances = self._euclidean(X, self.centroids_)
        return np.argmin(distances, axis=1)

    def fit_predict(self, X):
        return self.fit(X).labels_


print("KMeansScratch class defined ✓")
print("Methods: fit(), predict(), fit_predict()")


---
## Cell 4 — Scikit-Learn Implementation

Scikit-Learn's `KMeans` is a production-ready implementation that adds:
- **K-Means++ initialization** — smarter centroid seeding that spreads initial centroids apart, reducing bad local minima
- **Multiple restarts** (`n_init`) — runs the full algorithm N times, keeps the best result
- **Optimized C-level routines** — 10–100× faster than pure Python on large datasets
- **Mini-batch variant** (`MiniBatchKMeans`) — for datasets that don't fit in memory


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ── Select features for clustering ───────────────────────────────────────────
# We use Annual Income and Spending Score — the two most insightful axes
features = ['Annual Income (k$)', 'Spending Score (1-100)']
X_raw = df[features].values

# ── Scale features ────────────────────────────────────────────────────────────
# StandardScaler: zero mean, unit variance
# Critical: distance-based algorithms are sensitive to feature magnitude
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Raw feature ranges:")
for i, f in enumerate(features):
    print(f"  {f}: [{X_raw[:,i].min():.1f}, {X_raw[:,i].max():.1f}]")

print(f"\nScaled feature ranges (should be ~[-3, 3]):")
for i, f in enumerate(features):
    print(f"  {f}: [{X_scaled[:,i].min():.2f}, {X_scaled[:,i].max():.2f}]")


In [ ]:
# ── Train Scikit-Learn KMeans ─────────────────────────────────────────────────
sklearn_kmeans = KMeans(
    n_clusters=5,        # We'll justify K=5 via Elbow Method in Cell 8
    init='k-means++',    # Smart initialization — reduces bad convergence
    n_init=10,           # Run 10 times, keep best (lowest inertia)
    max_iter=300,
    random_state=42
)
sklearn_kmeans.fit(X_scaled)

print("── Scikit-Learn KMeans Results ──")
print(f"  Converged in : {sklearn_kmeans.n_iter_} iterations")
print(f"  Inertia      : {sklearn_kmeans.inertia_:.4f}")
print(f"  Cluster sizes: {dict(zip(*np.unique(sklearn_kmeans.labels_, return_counts=True)))}")
print(f"  Centroids (scaled):")
for i, c in enumerate(sklearn_kmeans.cluster_centers_):
    print(f"    Cluster {i}: {c.round(3)}")


---
## Cell 5 — Data Preprocessing

Good preprocessing is non-negotiable for distance-based algorithms like K-Means.

### Why Scaling Matters
Imagine clustering with `Annual Income (k$)` ranging 15–140 and `Age` ranging 18–70.
Income has ~3× the raw numeric range, so Euclidean distances are dominated by it —
age effectively becomes invisible. `StandardScaler` fixes this.

### Steps
1. **Check for missing values** — already done in Cell 1b (none found)
2. **Encode categoricals** — convert `Genre` to numeric (0/1)
3. **Scale numeric features** — zero mean, unit variance
4. **Prepare final feature matrix** for experiments


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ── Encode Genre ──────────────────────────────────────────────────────────────
df_processed = df.copy()
le = LabelEncoder()
df_processed['Genre_enc'] = le.fit_transform(df_processed['Genre'])  # Male=1, Female=0
print(f"Genre encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# ── Feature sets ──────────────────────────────────────────────────────────────
# For main experiments: Income vs Spending (most interpretable 2D view)
X_2d_raw    = df_processed[['Annual Income (k$)', 'Spending Score (1-100)']].values

# For richer experiments: Age + Income + Spending
X_3d_raw    = df_processed[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].values

# Scale both
scaler_2d = StandardScaler()
X_2d      = scaler_2d.fit_transform(X_2d_raw)

scaler_3d = StandardScaler()
X_3d      = scaler_3d.fit_transform(X_3d_raw)

print(f"\n2D feature matrix shape : {X_2d.shape}  (Income + Spending)")
print(f"3D feature matrix shape : {X_3d.shape}  (Age + Income + Spending)")
print(f"\nPreprocessing complete ✓")
print(f"  • Categorical encoding : Genre → 0/1")
print(f"  • Scaling              : StandardScaler (mean=0, std=1)")
print(f"  • Missing values       : None detected")


---
## Cell 6 — Training & Evaluation

### Evaluation Metrics

| Metric | What It Measures | Better When |
|--------|-----------------|-------------|
| **Inertia (WCSS)** | Total squared distance of points to their centroid | Lower |
| **Silhouette Score** | How well-separated clusters are (-1 to +1) | Higher (closer to +1) |

### Silhouette Score
For each point `i`:
$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\ b(i))}$$

- `a(i)` = mean distance to all other points **in the same cluster** (cohesion)
- `b(i)` = mean distance to all points in the **nearest other cluster** (separation)
- Score of **+1** = perfectly assigned; **0** = on boundary; **-1** = wrong cluster

### From-Scratch vs Sklearn Comparison
We run both implementations on the same data and compare inertia, labels, and wall-clock time.


In [ ]:
import time
from sklearn.metrics import silhouette_score

K = 5   # Justified by Elbow Method (see Cell 8)

# ── Run FROM SCRATCH ─────────────────────────────────────────────────────────
print("── From-Scratch K-Means ──")
t0 = time.time()
scratch_km = KMeansScratch(n_clusters=K, max_iter=300, tol=1e-4, random_state=42)
scratch_km.fit(X_2d)
t_scratch = time.time() - t0

sil_scratch = silhouette_score(X_2d, scratch_km.labels_)
print(f"  Inertia         : {scratch_km.inertia_:.4f}")
print(f"  Silhouette Score: {sil_scratch:.4f}")
print(f"  Wall-clock time : {t_scratch*1000:.2f} ms")
print(f"  Cluster sizes   : {dict(zip(*np.unique(scratch_km.labels_, return_counts=True)))}")

print()

# ── Run SKLEARN ──────────────────────────────────────────────────────────────
print("── Scikit-Learn KMeans ──")
t0 = time.time()
sk_km = KMeans(n_clusters=K, init='k-means++', n_init=10, max_iter=300, random_state=42)
sk_km.fit(X_2d)
t_sklearn = time.time() - t0

sil_sklearn = silhouette_score(X_2d, sk_km.labels_)
print(f"  Inertia         : {sk_km.inertia_:.4f}")
print(f"  Silhouette Score: {sil_sklearn:.4f}")
print(f"  Wall-clock time : {t_sklearn*1000:.2f} ms")
print(f"  Cluster sizes   : {dict(zip(*np.unique(sk_km.labels_, return_counts=True)))}")

print()

# ── Comparison Summary ────────────────────────────────────────────────────────
print("── Comparison Summary ──")
print(f"{'Metric':<25} {'Scratch':>12} {'Sklearn':>12}")
print("-" * 50)
print(f"{'Inertia':<25} {scratch_km.inertia_:>12.4f} {sk_km.inertia_:>12.4f}")
print(f"{'Silhouette Score':<25} {sil_scratch:>12.4f} {sil_sklearn:>12.4f}")
print(f"{'Time (ms)':<25} {t_scratch*1000:>12.2f} {t_sklearn*1000:>12.2f}")
print(f"{'Iterations':<25} {scratch_km.n_iter_:>12d} {sk_km.n_iter_:>12d}")


---
## Cell 7 — Visualization

Two essential plots for K-Means results:

1. **Cluster Scatter Plot** — shows the 2D distribution of data points colored by cluster, with centroids marked
2. **Scratch vs Sklearn Side-by-Side** — confirms both implementations agree on the cluster structure

> **Reading the plot:** Centroids (★) should sit near the geometric center of each colored group.
> Overlapping regions at cluster boundaries are expected — K-Means makes hard assignments.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Color palette ─────────────────────────────────────────────────────────────
COLORS  = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#9B5DE5']
PALETTE = [COLORS[i] for i in range(K)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('K-Means Clustering — Mall Customer Segmentation', fontsize=15, fontweight='bold', y=1.01)

feature_names = ['Annual Income (k$)', 'Spending Score (1–100)']

for ax, (labels, centroids, title) in zip(axes, [
    (scratch_km.labels_,  scratch_km.centroids_,      'From Scratch (NumPy)'),
    (sk_km.labels_,       sk_km.cluster_centers_,     'Scikit-Learn (k-means++)')
]):
    # Scatter points
    for k in range(K):
        mask = labels == k
        ax.scatter(
            X_2d[mask, 0], X_2d[mask, 1],
            c=PALETTE[k], alpha=0.65, s=60, edgecolors='white', linewidths=0.4,
            label=f'Cluster {k} (n={mask.sum()})'
        )
    # Centroids
    ax.scatter(
        centroids[:, 0], centroids[:, 1],
        c='black', marker='*', s=280, zorder=5, label='Centroids'
    )
    ax.set_xlabel('Annual Income (scaled)', fontsize=11)
    ax.set_ylabel('Spending Score (scaled)', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved → kmeans_clusters.png")


In [ ]:
# ── Visualization 2: Cluster Profiles (Mean Feature Values per Cluster) ──────
# Inverse-transform scaled centroids back to original feature space for interpretation

centroids_orig = scaler_2d.inverse_transform(sk_km.cluster_centers_)
cluster_labels  = [f'Cluster {i}' for i in range(K)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cluster Centroid Profiles — Original Feature Space', fontsize=14, fontweight='bold')

for i, (ax, feat, idx) in enumerate(zip(axes, feature_names, [0, 1])):
    bars = ax.bar(cluster_labels, centroids_orig[:, idx],
                  color=PALETTE, edgecolor='black', alpha=0.85)
    ax.set_title(f'Mean {feat} per Cluster', fontsize=11)
    ax.set_ylabel(feat)
    ax.set_ylim(0, centroids_orig[:, idx].max() * 1.2)
    for bar, val in zip(bars, centroids_orig[:, idx]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('kmeans_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCluster Centroid Interpretation (original scale):")
print(f"{'Cluster':<12} {'Avg Income (k$)':>18} {'Avg Spending Score':>20}")
print("-" * 52)
for i, (inc, spd) in enumerate(centroids_orig):
    print(f"Cluster {i:<4}  {inc:>18.1f} {spd:>20.1f}")


---
## Cell 8 — Hyperparameter Experiments

### Finding the Optimal K

Two complementary methods:

**1. Elbow Method** — Plot inertia vs K. Look for the "elbow" point where adding more clusters yields diminishing returns. The bend = a natural plateau in tightness.

**2. Silhouette Score** — Plot silhouette vs K. The K with the highest silhouette score has the best-separated, most cohesive clusters. More principled than the elbow alone.

> **Best practice:** Use *both* — the elbow tells you where improvement flattens; silhouette tells you where separation is maximized. When they agree, you've found your K.


In [ ]:
from sklearn.metrics import silhouette_score

K_range    = range(2, 11)
inertias   = []
silhouettes = []

print("Running K-Means for K = 2 to 10...")
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=300, random_state=42)
    km.fit(X_2d)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_2d, km.labels_)
    silhouettes.append(sil)
    print(f"  K={k:2d} | Inertia={km.inertia_:8.2f} | Silhouette={sil:.4f}")

best_k_sil = list(K_range)[np.argmax(silhouettes)]
print(f"\n  Best K by Silhouette Score: K = {best_k_sil}")


In [ ]:
# ── Plot Elbow + Silhouette ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Hyperparameter Selection: Optimal K', fontsize=14, fontweight='bold')

K_list = list(K_range)

# Elbow Method
ax1 = axes[0]
ax1.plot(K_list, inertias, 'o-', color='#E63946', linewidth=2.5, markersize=8)
ax1.axvline(x=5, color='#457B9D', linestyle='--', linewidth=2, alpha=0.8, label='K=5 (elbow)')
ax1.fill_between(K_list, inertias, alpha=0.08, color='#E63946')
ax1.set_title('Elbow Method — Inertia vs K', fontsize=12, fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)', fontsize=11)
ax1.set_ylabel('Inertia (WCSS)', fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Annotate the elbow
ax1.annotate('Elbow ≈ K=5', xy=(5, inertias[3]), xytext=(7, inertias[3]*1.05),
             arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)

# Silhouette Score
ax2 = axes[1]
bar_colors = ['#2A9D8F' if k == best_k_sil else '#A8DADC' for k in K_list]
bars = ax2.bar(K_list, silhouettes, color=bar_colors, edgecolor='black', alpha=0.85)
ax2.axvline(x=best_k_sil, color='#E63946', linestyle='--', linewidth=2,
            label=f'Best K={best_k_sil}')
ax2.set_title('Silhouette Score vs K', fontsize=12, fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)', fontsize=11)
ax2.set_ylabel('Silhouette Score', fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, silhouettes):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('kmeans_hyperparams.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nConclusion: Both methods point to K=5 as the optimal cluster count.")


---
## Cell 9 — Interview Corner 🎤

The five K-Means questions that appear most frequently in ML engineering interviews at FAANG companies — with model answers that demonstrate both theory and production judgment.

---

### Q1: What does K-Means optimize, and is it guaranteed to find the global optimum?

**Model Answer:**  
K-Means minimizes the Within-Cluster Sum of Squares (WCSS / Inertia):

$$J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \| x_i - \mu_k \|^2$$

It is **not** guaranteed to find the global minimum. The algorithm converges to a local minimum that depends on initialization. Mitigation: use **K-Means++ initialization** and **`n_init > 1`** (run multiple times, keep the best result).

---

### Q2: What is K-Means++ and why does it matter?

**Model Answer:**  
Standard K-Means picks initial centroids uniformly at random, which can cluster multiple centroids into the same dense region and produce poor partitions. K-Means++ seeds centroids probabilistically: each new centroid is chosen with probability proportional to its squared distance from the nearest already-chosen centroid. This spreads centroids apart from the start, leading to faster convergence, lower final inertia, and fewer restarts needed.

---

### Q3: How do you choose the optimal K?

**Model Answer:**  
Two complementary techniques:
1. **Elbow Method** — Plot WCSS vs K; pick the K at the "elbow" where the rate of improvement sharply decreases.
2. **Silhouette Score** — For each K, compute `(b - a) / max(a, b)` where `a` = intra-cluster cohesion and `b` = inter-cluster separation. Pick K that maximizes this score.

Domain knowledge should always inform or constrain the selection — if you know you're segmenting customers into budget tiers, K=4 or 5 is a natural prior.

---

### Q4: What are K-Means' key failure modes in production?

**Model Answer:**  
| Failure Mode | Cause | Fix |
|-------------|-------|-----|
| Feature dominance | Unscaled features | `StandardScaler` before clustering |
| Elongated clusters misclassified | Non-spherical cluster shapes | Use GMM or DBSCAN |
| Sensitive to outliers | Outliers pull centroids away from true center | Cap outliers or use K-Medoids |
| Different cluster sizes merged | Equal-size assumption | Use GMM or check with silhouette per cluster |
| Bad local minimum | Poor random initialization | K-Means++ + `n_init=10+` |

---

### Q5: What is the time complexity of K-Means? When does it fail to scale?

**Model Answer:**  
K-Means is **O(n · K · I · d)** where `n` = points, `K` = clusters, `I` = iterations, `d` = dimensions.
- Scales **linearly in n** — handles millions of points with mini-batch variants (`MiniBatchKMeans`)
- Fails with **very high d** — Euclidean distance becomes uninformative (curse of dimensionality); apply PCA first
- Fails with **very large K** — the assignment step becomes expensive; consider hierarchical clustering for large K scenarios


---
## Cell 10 — Key Takeaways ✅

| # | Takeaway |
|---|----------|
| 1 | **Always scale your features** before running K-Means — Euclidean distance is scale-sensitive |
| 2 | **K-Means++ initialization** (`init='k-means++'`) is better than random in virtually every case |
| 3 | **Use `n_init=10+`** to escape local minima — run multiple times, keep the best inertia |
| 4 | **The Elbow Method + Silhouette Score together** give a reliable K estimate; never rely on just one |
| 5 | **K-Means assumes spherical, roughly equal-sized clusters** — for irregular shapes, use DBSCAN or GMM |
| 6 | **From-scratch implementation** in NumPy is slower but teaches you exactly what the algorithm does; use Scikit-Learn in production |
| 7 | **Inertia always decreases with K** — it can't be used alone; you need silhouette or domain knowledge |
| 8 | **Outliers distort centroids** — clean or cap outliers before clustering, or use K-Medoids |
| 9 | **For large datasets**, use `MiniBatchKMeans` — same API, dramatically faster, slight quality tradeoff |
| 10 | **Interpret your clusters** by inverse-transforming centroids back to original scale — the numbers should make business sense |

---

### When to Reach for Alternatives

```
Clusters have arbitrary shapes?          → DBSCAN / HDBSCAN
Clusters have different densities?       → HDBSCAN
Need soft/probabilistic membership?      → Gaussian Mixture Models (GMM)
Need a full cluster hierarchy?           → Agglomerative / Hierarchical Clustering
Dataset > 10M rows, latency critical?    → MiniBatchKMeans or approximate methods
```


---
## Cell 11 — References & Further Reading 📚

| Resource | Link |
|----------|------|
| **Dataset** — Mall Customer Segmentation | [Kaggle](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python) |
| **Original K-Means Paper** — MacQueen (1967) | [CMU PDF](https://www.cs.cmu.edu/~bhiksha/courses/mlsp.fall2010/class14/macqueen.pdf) |
| **K-Means++ Paper** — Arthur & Vassilvitskii (2007) | [Stanford](http://ilpubs.stanford.edu:8090/778/1/2006-13.pdf) |
| **Scikit-Learn KMeans Docs** | [sklearn.org](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) |
| **StatQuest Visual Explainer** | [YouTube](https://www.youtube.com/watch?v=4b5d3muPQmA) |
| **Kaggle Notebook — Customer Segmentation** | [Kaggle](https://www.kaggle.com/code/kushal1996/customer-segmentation-k-means-analysis) |

---

*Notebook authored in the style of MIT 6.867 Machine Learning curriculum.*  
*Next in series: → GMM & Expectation Maximization | → DBSCAN & HDBSCAN | → Hierarchical Clustering*
